# Built-in Tools: Bash Execution

The SDK's built-in `Bash` tool lets the agent run shell commands directly — useful for tasks like checking file counts, running scripts, or inspecting the environment.


In [3]:
from pathlib import Path
from claude_agent_sdk import (
    ClaudeSDKClient,  # a client you can keep open and send several messages through
    ClaudeAgentOptions,  # settings object: model, system prompt, tools, etc.
    AssistantMessage,  # message type that holds Claude's actual reply
    ToolUseBlock,  # message piece that shows "Claude is calling a tool now"
    ResultMessage,  # the last message in the stream — carries the final answer plus stats (cost, duration, etc.)
)


async def run_bash_demo() -> None:
    # allowed_tools=["Bash"]: only pre-approve the Bash tool (run shell commands).
    # Claude can't use Read/Write/Edit/etc. here — just Bash.
    options = ClaudeAgentOptions(model="haiku", allowed_tools=["Bash"])
    async with ClaudeSDKClient(options=options) as client:
        scratch_dir = Path.cwd() / "05_scratch"
        scratch_dir.mkdir(parents=True, exist_ok=True)
        await client.query(f"Using bash, create a file name notes.txt in the {scratch_dir} directory.")

        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    # ToolUseBlock shows up whenever Claude decides to call a tool.
                    # block.name  -> which tool ("Bash")
                    # block.input -> the arguments it's calling it with (the command)
                    if isinstance(block, ToolUseBlock):
                        print(f"[tool call] {block.name}({block.input})")
            elif isinstance(message, ResultMessage):
                print(f"\nResult: {message.result}")


await run_bash_demo()

[tool call] Bash({'command': 'touch /Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/episodes/05_scratch/notes.txt', 'description': 'Create notes.txt file in the specified directory'})
[tool call] Bash({'command': 'ls -la /Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/episodes/05_scratch/notes.txt', 'description': 'Verify the notes.txt file was created'})

Result: Perfect! I've successfully created the `notes.txt` file in `/Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/episodes/05_scratch/`. 

The file was created as an empty file with read/write permissions for the owner and read-only permissions for group and others.


## Safety note

`allowed_tools=["Bash"]` auto-approves every Bash call in this demo, which is fine for a scoped, throwaway notebook — but in production you'll want to scope or permission Bash execution much more carefully. The Permissions episode later in this series covers exactly that.


In [2]:
import shutil
from pathlib import Path

# Scratchpad cleanup — remove any files Claude created, then the directory itself.
scratch_dir = Path.cwd() / "05_scratch"

if scratch_dir.exists():
    remaining = [p.name for p in scratch_dir.iterdir()]
    print("05_scratch contents:", remaining)
    shutil.rmtree(scratch_dir)
    print(f"Removed {scratch_dir}")
else:
    print("05_scratch does not exist — nothing to clean up.")

05_scratch contents: ['notes.txt']
Removed /Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/episodes/05_scratch
